# Step 7 — Synthetic Data Generation (SMOTE)

This notebook generates synthetic samples via SMOTE to augment minority classes,
then compares model performance on original vs augmented data.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score

from asd_pipeline_utils import (
    RANDOM_STATE, SMOTE_PFI_PATH,
    get_targets, load_analysis_frame, split_features_and_metadata,
    generate_smote_samples,
)

final_df = load_analysis_frame()
X, meta = split_features_and_metadata(final_df)
_, y_multi = get_targets(meta)

# Use PFI-selected genes from Step 6
pfi_df = pd.read_csv(SMOTE_PFI_PATH)
top_pfi_genes = pfi_df[pfi_df["importance_mean"] > 0]["gene"].tolist()
if len(top_pfi_genes) < 20:
    top_pfi_genes = pfi_df.head(30)["gene"].tolist()

print(f"Step 7: Using {len(top_pfi_genes)} PFI-selected genes")
print(f"\nOriginal class distribution:\n{y_multi.value_counts()}")

# Generate SMOTE samples
X_synth = X[top_pfi_genes].copy()
X_res, y_res, scaler, imputer = generate_smote_samples(X_synth, y_multi)

print(f"\nAfter SMOTE: {X_synth.shape[0]} -> {X_res.shape[0]} samples (+{X_res.shape[0] - X_synth.shape[0]} synthetic)")
print(f"Resampled distribution:\n{pd.Series(y_res).value_counts()}")

# Compare: original vs augmented
X_sc = pd.DataFrame(scaler.transform(imputer.transform(X_synth)), index=X_synth.index, columns=X_synth.columns)

clf_orig = LogisticRegression(solver="saga", max_iter=8000, random_state=RANDOM_STATE, class_weight="balanced")
scores_orig = cross_val_score(clf_orig, X_sc, y_multi, cv=5, scoring="balanced_accuracy")

clf_smote = LogisticRegression(solver="saga", max_iter=8000, random_state=RANDOM_STATE)
scores_smote = cross_val_score(clf_smote, X_res, y_res, cv=5, scoring="balanced_accuracy")

print(f"\n=== Performance Comparison ===")
print(f"Original (class_weight=balanced): bal_acc = {scores_orig.mean():.3f} +/- {scores_orig.std():.3f}")
print(f"SMOTE-augmented:                  bal_acc = {scores_smote.mean():.3f} +/- {scores_smote.std():.3f}")

# Held-out evaluation (SMOTE on train only)
X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y_multi, test_size=0.2, stratify=y_multi, random_state=RANDOM_STATE)
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=RANDOM_STATE)
X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)

clf = LogisticRegression(solver="saga", max_iter=8000, random_state=RANDOM_STATE)
clf.fit(X_tr_res, y_tr_res)
y_pred = clf.predict(X_te)

print(f"\n=== Held-out Test (SMOTE on train only) ===")
print(f"Balanced accuracy: {balanced_accuracy_score(y_te, y_pred):.3f}")
print(f"\n{classification_report(y_te, y_pred)}")